# Stress Classification v3 — Multi-Model Ensemble + SMOTE

Key improvements over v2:
- **4-model soft-vote ensemble**: LightGBM + XGBoost + CatBoost + Random Forest
- **SMOTE oversampling**: synthesizes class 1 samples (only 66 real samples)
- **Richer features**: third-window means, t3−t1 delta, CV, EDA×HR cross features
- **LOPO CV** as primary metric (Leave-One-Person-Out)

In [1]:
%pip install lightgbm xgboost catboost scikit-learn pandas numpy scipy imbalanced-learn -q


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

TRAIN_DATA  = pd.read_csv('train-sensor.csv')
TRAIN_LABEL = pd.read_csv('train-label.csv')
TEST_DATA   = pd.read_csv('test-sensor.csv')
TEST_LABEL  = pd.read_csv('test-label.csv')

TRAIN_LABEL['timestamp'] = TRAIN_LABEL['timestamp'].astype(float)
TEST_LABEL['timestamp']  = TEST_LABEL['timestamp'].astype(str).str.strip().astype(float)

print('TRAIN_DATA shape :', TRAIN_DATA.shape)
print('TRAIN_LABEL shape:', TRAIN_LABEL.shape)
print('TEST_DATA shape  :', TEST_DATA.shape)
print('TEST_LABEL shape :', TEST_LABEL.shape)
print()
print('Stress distribution:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())


TRAIN_DATA shape : (4694400, 8)
TRAIN_LABEL shape: (815, 4)
TEST_DATA shape  : (5921280, 8)
TEST_LABEL shape : (1028, 4)

Stress distribution:
stress
0.0    162
1.0     66
2.0    587
Name: count, dtype: int64


## 1. Libraries

In [3]:
from scipy import stats as spstats
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.metrics import balanced_accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter


## 2. Per-Person Z-Score Normalisation

EDA varies 60x across people. We z-score each sensor by that person's own mean/std
so features represent *relative deviation from baseline*, not absolute values.
For test: use each test person's own sensor recording as their baseline.

In [4]:
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']

def compute_pid_stats(sensor_df):
    d = {}
    for pid, grp in sensor_df.groupby('pid'):
        d[pid] = {}
        for c in SENSOR_COLS:
            v  = grp[c].dropna().values.astype(float)
            mu = float(np.mean(v)) if len(v) > 0 else 0.0
            sd = float(np.std(v))  if len(v) > 0 else 1.0
            d[pid][c + '_mu'] = mu
            d[pid][c + '_sd'] = max(sd, 1e-6)
    return d

train_pid_stats = compute_pid_stats(TRAIN_DATA)
test_pid_stats  = compute_pid_stats(TEST_DATA)

global_mu = {c: np.median([train_pid_stats[p][c+'_mu'] for p in train_pid_stats]) for c in SENSOR_COLS}
global_sd = {c: np.median([train_pid_stats[p][c+'_sd'] for p in train_pid_stats]) for c in SENSOR_COLS}

print('Per-person EDA baselines (train):')
for pid in sorted(train_pid_stats):
    print(f'  {pid}: eda={train_pid_stats[pid]["eda_mu"]:.3f}  temp={train_pid_stats[pid]["temperature_mu"]:.3f}')


Per-person EDA baselines (train):
  43JW: eda=2.072  temp=33.377
  C8Q6: eda=0.127  temp=29.952
  DT5C: eda=4.531  temp=33.462
  F1ZM: eda=0.094  temp=29.457
  HDS9: eda=5.719  temp=30.171
  P4DZ: eda=1.183  temp=29.567
  TPQI: eda=3.551  temp=32.924


## 3. Rich Feature Extraction

For each 3-min label window:
- **11 z-scored stats** per sensor (mean/std/min/max/median/skew/kurt/range/q25/q75/iqr)
- **delta**: last-half mean minus first-half mean — captures within-window change direction
- **slope**: linear trend across full window
- **t1_mean / t3_mean**: mean of first 60s vs last 60s (finer temporal resolution)
- **t3t1_delta**: t3_mean - t1_mean — stress trajectory over 3 segments
- **cv**: coefficient of variation (std/|mean|) of raw signal — personal volatility
- **eda_hr_product**: EDA x HR z-score product — both rise together during real stress
- **eda_hr_corr**: correlation between EDA and HR in the window

In [5]:
WINDOW_MS = 180_000   # 3 minutes
HALF_MS   =  90_000   # 1.5 min for delta
THIRD_MS  =  60_000   # 1 min for third-window features

def znorm(col_series, mu, sd):
    v = col_series.dropna().values.astype(float)
    return (v - mu) / sd if len(v) > 0 else np.array([])

def extract_features(label_df, sensor_df, pid_stats):
    sensor_by_pid = {
        pid: grp.sort_values('timestamp').reset_index(drop=True)
        for pid, grp in sensor_df.groupby('pid')
    }

    rows = []
    for _, lrow in label_df.iterrows():
        pid = lrow['pid']
        ts  = float(lrow['timestamp'])
        lid = lrow['id']
        feat = {'id': lid}

        if pid not in sensor_by_pid:
            rows.append(feat)
            continue

        sg     = sensor_by_pid[pid]
        ta     = sg['timestamp'].values

        wa  = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts),             SENSOR_COLS]
        wf  = sg.loc[(ta >= ts - WINDOW_MS) & (ta <  ts - HALF_MS),  SENSOR_COLS]
        wl  = sg.loc[(ta >= ts - HALF_MS)   & (ta <= ts),            SENSOR_COLS]
        wt1 = sg.loc[(ta >= ts - WINDOW_MS) & (ta <  ts - 2*THIRD_MS), SENSOR_COLS]
        wt3 = sg.loc[(ta >= ts - THIRD_MS)  & (ta <= ts),            SENSOR_COLS]

        for c in SENSOR_COLS:
            mu = pid_stats.get(pid, {}).get(c + '_mu', global_mu[c])
            sd = pid_stats.get(pid, {}).get(c + '_sd', global_sd[c])

            va  = znorm(wa[c],  mu, sd)
            vf  = znorm(wf[c],  mu, sd)
            vl  = znorm(wl[c],  mu, sd)
            vt1 = znorm(wt1[c], mu, sd)
            vt3 = znorm(wt3[c], mu, sd)

            if len(va) == 0:
                for s in ['mean','std','min','max','median','skew','kurt','range',
                          'q25','q75','iqr','delta','slope','t1_mean','t3_mean',
                          't3t1_delta','cv']:
                    feat[f'{c}_{s}'] = np.nan
                continue

            # Standard z-scored stats
            feat[f'{c}_mean']   = float(np.mean(va))
            feat[f'{c}_std']    = float(np.std(va))
            feat[f'{c}_min']    = float(np.min(va))
            feat[f'{c}_max']    = float(np.max(va))
            feat[f'{c}_median'] = float(np.median(va))
            feat[f'{c}_skew']   = float(spstats.skew(va))     if len(va) > 2 else 0.0
            feat[f'{c}_kurt']   = float(spstats.kurtosis(va)) if len(va) > 2 else 0.0
            feat[f'{c}_range']  = float(np.max(va) - np.min(va))
            feat[f'{c}_q25']    = float(np.percentile(va, 25))
            feat[f'{c}_q75']    = float(np.percentile(va, 75))
            feat[f'{c}_iqr']    = float(np.percentile(va, 75) - np.percentile(va, 25))

            # Delta (last 90s - first 90s)
            feat[f'{c}_delta'] = float(np.mean(vl) - np.mean(vf)) if len(vf)>0 and len(vl)>0 else 0.0

            # Slope across full window
            feat[f'{c}_slope'] = float(np.polyfit(np.linspace(0,1,len(va)), va, 1)[0]) if len(va)>2 else 0.0

            # Third-window means
            feat[f'{c}_t1_mean']    = float(np.mean(vt1)) if len(vt1) > 0 else 0.0
            feat[f'{c}_t3_mean']    = float(np.mean(vt3)) if len(vt3) > 0 else 0.0
            feat[f'{c}_t3t1_delta'] = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']

            # Coefficient of variation (raw signal — captures personal volatility)
            raw_v = wa[c].dropna().values.astype(float)
            raw_m = np.mean(raw_v)
            feat[f'{c}_cv'] = float(np.std(raw_v) / abs(raw_m)) if abs(raw_m) > 1e-6 else 0.0

        # Accel magnitude
        ax = wa['accel_x'].values; ay = wa['accel_y'].values; az = wa['accel_z'].values
        if len(ax) > 0:
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan

        # Cross-sensor: EDA x HR (both rise during sympathetic activation)
        emu = pid_stats.get(pid, {}).get('eda_mu',        global_mu['eda'])
        esd = pid_stats.get(pid, {}).get('eda_sd',        global_sd['eda'])
        hmu = pid_stats.get(pid, {}).get('heart_rate_mu', global_mu['heart_rate'])
        hsd = pid_stats.get(pid, {}).get('heart_rate_sd', global_sd['heart_rate'])
        ez  = znorm(wa['eda'],        emu, esd)
        hz  = znorm(wa['heart_rate'], hmu, hsd)
        if len(ez) > 2 and len(hz) > 2:
            n = min(len(ez), len(hz))
            feat['eda_hr_product'] = float(np.mean(ez[:n] * hz[:n]))
            feat['eda_hr_corr']    = float(np.corrcoef(ez[:n], hz[:n])[0, 1])
        else:
            feat['eda_hr_product'] = feat['eda_hr_corr'] = 0.0

        rows.append(feat)

    return pd.DataFrame(rows).set_index('id')


print('Extracting train features...')
train_features = extract_features(TRAIN_LABEL, TRAIN_DATA, train_pid_stats)
print(f'  shape: {train_features.shape}')

print('Extracting test features...')
test_features = extract_features(TEST_LABEL, TEST_DATA, test_pid_stats)
print(f'  shape: {test_features.shape}')


Extracting train features...
  shape: (815, 107)
Extracting test features...
  shape: (1028, 107)


## 4. Prepare X, y

In [6]:
tli = TRAIN_LABEL.set_index('id')
y      = tli['stress'].astype(int)
groups = tli['pid']          # used for LOPO CV

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features),
                           columns=train_features.columns, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),
                           columns=test_features.columns, index=test_features.index)

print('X_imp shape     :', X_imp.shape)
print('X_test_imp shape:', X_test_imp.shape)
print('Class distribution:', y.value_counts().sort_index().to_dict())

counts        = Counter(y)
total         = len(y)
n_cls         = len(counts)
class_weights = {c: total / (n_cls * cnt) for c, cnt in counts.items()}
sample_weights = np.array([class_weights[yi] for yi in y])
print('Class weights:', {k: round(v,3) for k,v in class_weights.items()})


X_imp shape     : (815, 107)
X_test_imp shape: (1028, 107)
Class distribution: {0: 162, 1: 66, 2: 587}
Class weights: {1: 4.116, 0: 1.677, 2: 0.463}


## 5. Model Definitions

Four complementary models based on course slides:
- **LightGBM** — fast leaf-wise boosting, best for tabular (GOSS)
- **XGBoost** — level-wise boosting with L1/L2 regularisation
- **CatBoost** — ordered boosting, strong anti-overfitting (Oblivious Trees)
- **Random Forest** — bagging + random subspace (reduces variance independently)

Each model sees SMOTE-augmented data. Final prediction = soft-vote of all 4 probabilities.

In [7]:
LGBM_PARAMS = dict(
    n_estimators=1000, learning_rate=0.02, num_leaves=63,
    min_child_samples=5, subsample=0.7, colsample_bytree=0.7,
    reg_alpha=0.3, reg_lambda=0.3, class_weight='balanced',
    objective='multiclass', num_class=3, n_jobs=-1, verbose=-1,
)

XGB_PARAMS = dict(
    n_estimators=500, learning_rate=0.03, max_depth=5,
    subsample=0.7, colsample_bytree=0.7, reg_alpha=0.3, reg_lambda=1.0,
    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
    n_jobs=-1, verbosity=0,
)

RF_PARAMS = dict(
    n_estimators=500, max_depth=10, min_samples_leaf=3,
    class_weight='balanced', n_jobs=-1,
)

def make_smote(X_tr, y_tr):
    """SMOTE with safe k_neighbors — handles tiny class 1 folds."""
    cnt1 = Counter(y_tr).get(1, 0)
    if cnt1 < 2:
        return X_tr, y_tr
    k = max(1, min(5, cnt1 - 1))
    sm = SMOTE(k_neighbors=k, random_state=42)
    return sm.fit_resample(X_tr, y_tr)

def get_sw(y_arr):
    return np.array([class_weights[yi] for yi in y_arr])

print('Model parameters set.')


Model parameters set.


## 6. Leave-One-Person-Out CV (Real Generalization Estimate)

In [8]:
print('=== LOPO CV — 4-model ensemble + SMOTE ===')
logo = LeaveOneGroupOut()
lopo_all  = []
lopo_lgbm = []
lopo_xgb  = []
lopo_rf   = []
lopo_cat  = []

for tr_idx, val_idx in logo.split(X_imp, y, groups):
    pid_val = groups.iloc[val_idx[0]]
    y_val   = y.iloc[val_idx]

    if len(y_val.unique()) < 2:
        print(f'  Skip {pid_val}: only 1 class')
        continue

    X_tr = X_imp.iloc[tr_idx].values
    y_tr = y.iloc[tr_idx].values
    X_va = X_imp.iloc[val_idx].values

    X_sm, y_sm = make_smote(X_tr, y_tr)
    sw_sm = get_sw(y_sm)

    # LightGBM
    m1 = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': 42})
    m1.fit(X_sm, y_sm, sample_weight=sw_sm,
           eval_set=[(X_va, y_val)],
           callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    p1 = m1.predict_proba(X_va)

    # XGBoost
    m2 = xgb.XGBClassifier(**{**XGB_PARAMS, 'random_state': 42})
    m2.fit(X_sm, y_sm, sample_weight=sw_sm, verbose=False)
    p2 = m2.predict_proba(X_va)

    # Random Forest
    m3 = RandomForestClassifier(**{**RF_PARAMS, 'random_state': 42})
    m3.fit(X_sm, y_sm, sample_weight=sw_sm)
    p3 = m3.predict_proba(X_va)

    # CatBoost
    m4 = CatBoostClassifier(iterations=400, learning_rate=0.03, depth=6,
                             loss_function='MultiClass', auto_class_weights='Balanced',
                             verbose=False, random_seed=42)
    m4.fit(X_sm, y_sm, sample_weight=sw_sm)
    p4 = m4.predict_proba(X_va)

    # Soft-vote
    p_ens = (p1 + p2 + p3 + p4) / 4.0

    s1  = balanced_accuracy_score(y_val, np.argmax(p1,   1))
    s2  = balanced_accuracy_score(y_val, np.argmax(p2,   1))
    s3  = balanced_accuracy_score(y_val, np.argmax(p3,   1))
    s4  = balanced_accuracy_score(y_val, np.argmax(p4,   1))
    se  = balanced_accuracy_score(y_val, np.argmax(p_ens,1))

    lopo_lgbm.append(s1); lopo_xgb.append(s2)
    lopo_rf.append(s3);   lopo_cat.append(s4); lopo_all.append(se)

    print(f'  Leave out {pid_val}: lgbm={s1:.3f} xgb={s2:.3f} rf={s3:.3f} cat={s4:.3f} | ENS={se:.3f}')

print()
print(f'  LightGBM  LOPO = {np.mean(lopo_lgbm):.4f} +/- {np.std(lopo_lgbm):.4f}')
print(f'  XGBoost   LOPO = {np.mean(lopo_xgb):.4f} +/- {np.std(lopo_xgb):.4f}')
print(f'  RandForest LOPO = {np.mean(lopo_rf):.4f} +/- {np.std(lopo_rf):.4f}')
print(f'  CatBoost  LOPO = {np.mean(lopo_cat):.4f} +/- {np.std(lopo_cat):.4f}')
print(f'  ENSEMBLE  LOPO = {np.mean(lopo_all):.4f} +/- {np.std(lopo_all):.4f}')
print()
print(f'  >> Use ENSEMBLE LOPO as your primary metric for deciding to submit.')


=== LOPO CV — 4-model ensemble + SMOTE ===
  Leave out 43JW: lgbm=0.236 xgb=0.225 rf=0.214 cat=0.209 | ENS=0.220
  Leave out C8Q6: lgbm=0.454 xgb=0.508 rf=0.512 cat=0.472 | ENS=0.511
  Leave out DT5C: lgbm=0.334 xgb=0.437 rf=0.456 cat=0.372 | ENS=0.444
  Leave out F1ZM: lgbm=0.422 xgb=0.403 rf=0.366 cat=0.425 | ENS=0.410
  Leave out HDS9: lgbm=0.712 xgb=0.660 rf=0.662 cat=0.609 | ENS=0.677
  Leave out P4DZ: lgbm=0.331 xgb=0.295 rf=0.244 cat=0.296 | ENS=0.272
  Leave out TPQI: lgbm=0.469 xgb=0.493 rf=0.538 cat=0.455 | ENS=0.457

  LightGBM  LOPO = 0.4225 +/- 0.1402
  XGBoost   LOPO = 0.4316 +/- 0.1330
  RandForest LOPO = 0.4276 +/- 0.1504
  CatBoost  LOPO = 0.4055 +/- 0.1198
  ENSEMBLE  LOPO = 0.4275 +/- 0.1405

  >> Use ENSEMBLE LOPO as your primary metric for deciding to submit.


## 7. Final Ensemble Training

Train all 4 models across 5-fold splits on full data, aggregate test probabilities,
repeat for 3 seeds → final soft-vote across 3×5×4 = 60 model outputs.

In [9]:
print('=== Training final ensemble (3 seeds x 5 folds x 4 models) ===')
SEEDS = [42, 7, 123]
all_test_proba = []

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y)):
        X_tr = X_imp.iloc[tr_idx].values
        y_tr = y.iloc[tr_idx].values
        X_va = X_imp.iloc[val_idx].values
        y_va = y.iloc[val_idx]

        X_sm, y_sm = make_smote(X_tr, y_tr)
        sw_sm = get_sw(y_sm)

        m1 = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m1.fit(X_sm, y_sm, sample_weight=sw_sm,
               eval_set=[(X_va, y_va)],
               callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])

        m2 = xgb.XGBClassifier(**{**XGB_PARAMS, 'random_state': seed})
        m2.fit(X_sm, y_sm, sample_weight=sw_sm, verbose=False)

        m3 = RandomForestClassifier(**{**RF_PARAMS, 'random_state': seed})
        m3.fit(X_sm, y_sm, sample_weight=sw_sm)

        m4 = CatBoostClassifier(iterations=400, learning_rate=0.03, depth=6,
                                 loss_function='MultiClass', auto_class_weights='Balanced',
                                 verbose=False, random_seed=seed)
        m4.fit(X_sm, y_sm, sample_weight=sw_sm)

        fold_proba = (m1.predict_proba(X_test_imp.values) +
                      m2.predict_proba(X_test_imp.values) +
                      m3.predict_proba(X_test_imp.values) +
                      m4.predict_proba(X_test_imp.values)) / 4.0
        seed_proba += fold_proba

        val_preds = np.argmax(
            (m1.predict_proba(X_va) + m2.predict_proba(X_va) +
             m3.predict_proba(X_va) + m4.predict_proba(X_va)) / 4.0, axis=1)
        print(f'  Seed {seed} Fold {fold+1}: val BA = {balanced_accuracy_score(y_va, val_preds):.4f}')

    seed_proba /= 5
    all_test_proba.append(seed_proba)
    print(f'  Seed {seed} done.')

final_proba = np.mean(all_test_proba, axis=0)
final_preds = np.argmax(final_proba, axis=1).astype(int)

print()
print('Prediction distribution:')
for u, c in zip(*np.unique(final_preds, return_counts=True)):
    print(f'  class {u}: {c}')


=== Training final ensemble (3 seeds x 5 folds x 4 models) ===
  Seed 42 Fold 1: val BA = 0.7342
  Seed 42 Fold 2: val BA = 0.7247
  Seed 42 Fold 3: val BA = 0.7868
  Seed 42 Fold 4: val BA = 0.8182
  Seed 42 Fold 5: val BA = 0.8155
  Seed 42 done.
  Seed 7 Fold 1: val BA = 0.7608
  Seed 7 Fold 2: val BA = 0.7547
  Seed 7 Fold 3: val BA = 0.8438
  Seed 7 Fold 4: val BA = 0.8495
  Seed 7 Fold 5: val BA = 0.7509
  Seed 7 done.
  Seed 123 Fold 1: val BA = 0.8406
  Seed 123 Fold 2: val BA = 0.6661
  Seed 123 Fold 3: val BA = 0.8552
  Seed 123 Fold 4: val BA = 0.8315
  Seed 123 Fold 5: val BA = 0.7972
  Seed 123 done.

Prediction distribution:
  class 0: 235
  class 1: 35
  class 2: 758


# Submit

In [10]:
submission = pd.DataFrame({
    'id':     TEST_LABEL['id'].values,
    'stress': final_preds,
})
submission.to_csv('submission_v3.csv', index=False)
print('submission_v3.csv saved!')
print(submission.head(10))


submission_v3.csv saved!
     id  stress
0  1227       1
1  1228       2
2  1229       2
3  1230       2
4  1231       0
5  1232       2
6  1233       2
7  1234       2
8  1235       2
9  1236       2
